## Principles of Machine Learning Final Project
**By: Saheli Ray, Wyatt Golden, HuiDi Hu**

In [7]:
# All imports needed for the project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import warnings
warnings.filterwarnings("ignore")

pd.options.display.max_columns = 100
        


In [8]:
# Quick environment check
%cd /home/jovyan/work
!ls

/home/jovyan/work
decision_tree_submission.csv  random_forest_submission.csv  test.csv
FinalProject.ipynb	      README.md			    train.csv


## Data Exploration / Cleaning
**In this section of the notebook, we inspect the dataset, choose a focused feature set, and standardize the data so the same preparation logic can be reused for both training and test predictions.**
        


In [9]:
# Load dataset and look at its shape and first 5 rows
train = pd.read_csv("train.csv", low_memory=False)
print("Dataset shape:", train.shape)
train.head(5)

Dataset shape: (307178, 55)


,INDEX_NR,INCIDENT_DATE,INCIDENT_MONTH,INCIDENT_YEAR,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,LATITUDE,LONGITUDE,RUNWAY,STATE,FAAREGION,LOCATION,OPID,OPERATOR,REG,FLT,AIRCRAFT,AMA,AMO,EMA,EMO,AC_CLASS,AC_MASS,TYPE_ENG,NUM_ENGS,ENG_1_POS,ENG_2_POS,ENG_3_POS,ENG_4_POS,PHASE_OF_FLIGHT,HEIGHT,SPEED,DISTANCE,SKY,PRECIPITATION,BIRD_BAND_NUMBER,SPECIES_ID,SPECIES,OUT_OF_RANGE_SPECIES,REMARKS,REMAINS_COLLECTED,REMAINS_SENT,WARNED,NUM_SEEN,NUM_STRUCK,SIZE,ENROUTE_STATE,COMMENTS,SOURCE,PERSON,LUPDATE,TRANSFER,INDICATED_DAMAGE
0,1410120,12/13/93,12,1993,NaN,Day,TJSJ,LUIS MUNOZ MARIN INTL,18.43942,-66.00183,7,PR,ASO,NaN,AAL,AMERICAN AIRLINES,N892AA,NaN,B-727-200,148,11,34.0,10.0,A,4.0,D,3.0,5.0,6.0,5.0,NaN,Approach,300.0,145.0,NaN,Some Cloud,NaN,NaN,UNKBS,Unknown bird - small,0,NO SIGN OF BIRD ON A/C.,1,0,No,10-Feb,10-Feb,Small,NaN,NaN,FAA Form 5200-7,Pilot,4/3/23,0,0
1,709688,2/1/10,2,2010,5:00,Night,WMKK,KUALA LUMPUR INTL,2.745578,101.709917,32R,FN,FGN,NaN,FDX,FEDEX EXPRESS,N608FE,5293,MD-11,583,39,22.0,7.0,A,4.0,D,3.0,1.0,6.0,1.0,NaN,Approach,50.0,NaN,0.0,NaN,NaN,NaN,UNKBM,Unknown bird - medium,0,EVID OF STRIKE FOUND ON LOWER RT SIDE OF RADOME.,0,0,Unknown,NaN,1,Medium,NaN,2010-5-18-53374 /Legacy Record 300758/,FAA Form 5200-7-E,Air Transport Operations,6/9/10,0,0
2,730841,5/9/12,5,2012,2:00,Night,KSDF,MUHAMMAD ALI INTERNATIONAL,38.17439,-85.736,35L,KY,ASO,NaN,UPS,UPS AIRLINES,N141UP,907,A-300,04A,1,34.0,46.0,A,4.0,D,2.0,1.0,1.0,NaN,NaN,Approach,3500.0,240.0,8.0,NaN,NaN,NaN,UNKBL,Unknown bird - large,0,"STARTED TO SLOW DOWN FROM 250 KTS AT AROUND 4,...",0,0,No,NaN,1,Large,NaN,UPS EVENT REPT 36216 (4/22/13 UPDATED COST) /L...,Air Transport Report,Air Transport Operations,4/22/13,0,1
3,654676,10/8/02,10,2002,NaN,NaN,KLAX,LOS ANGELES INTL,33.94254,-118.40807,25R,CA,AWP,NaN,UNK,UNKNOWN,NaN,NaN,UNKNOWN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NE120,Western gull,0,REMAINS OF 2 GULLS WERE PICKED UP OF RWY.,1,0,Unknown,NaN,10-Feb,Medium,NaN,2002-10-8-111929 /Legacy Record 216397/,FAA Form 5200-7-E,Carcass Found,1/9/03,0,0
4,629708,2/3/97,2,1997,NaN,Dawn,PHLI,LIHUE ARPT,21.97598,-159.33896,35,HI,AWP,NaN,1AAH,ALOHA AIRLINES,NaN,NaN,B-737-200,148,13,34.0,10.0,A,4.0,D,2.0,1.0,1.0,NaN,NaN,Landing Roll,0.0,135.0,0.0,Some Cloud,NaN,NaN,R1101,American barn owl,0,TIME 0824 LCL.,0,0,No,1,1,Medium,NaN,SOURCE 5200-7 & PACIR /Legacy Record 121531/,Multiple,NaN,3/1/07,0,0


## Selected Features for Model Training

This dataset contains many variables, but for this modeling pass we chose fields that are both predictive and practical to clean consistently. The target variable `INDICATED_DAMAGE` is also included.

### Kept for the model
- `LATITUDE` / `LONGITUDE`: geographic position gives useful location signal without relying on high-cardinality airport codes.
- `AC_CLASS`, `AC_MASS`, `TYPE_ENG`: aircraft class, mass, and engine type capture differences in aircraft vulnerability.
- `SPECIES_ID`: a standardized wildlife code that is cleaner than free-text species descriptions.
- `SIZE`: animal size is closely related to how severe a strike may be.
- `SPEED`: aircraft speed at impact is important for damage severity.
- `NUM_STRUCK`: the number of animals struck may increase the chance of damage.
- `INDICATED_DAMAGE`: the target variable indicating whether damage occurred.

### Dropped by design for now
- `PHASE_OF_FLIGHT`: potentially useful, but left out in this pass to keep the baseline simpler.
- `HEIGHT`: omitted to avoid adding another sparse numeric field to the baseline.
- `SKY`: weather fields are less consistent and were not included in this model version.
- `INCIDENT_MONTH`, `TIME_OF_DAY`: temporal context may help later, but is not part of the current baseline feature set.

### How the current notebook handles missing data
- Numeric features are coerced to numeric values, and invalid entries become missing values.
- Blank categorical values are converted to missing values.
- Rows with missing `INDICATED_DAMAGE` are dropped from training because they cannot be used as labeled examples.
- Missing feature values are imputed inside the model pipeline so every row in the competition test set can still receive a prediction.
        


In [10]:
# These are the columns we will keep for modeling, including the target.
selected_columns = [
    "LATITUDE",
    "LONGITUDE",
    "AC_CLASS",
    "AC_MASS",
    "TYPE_ENG",
    "SPEED",
    "SPECIES_ID",
    "SIZE",
    "NUM_STRUCK",
    "INDICATED_DAMAGE",  # Target variable
]

# Split the selected columns into numeric features, categorical features, and the target-free feature list.
numeric_features = ["LATITUDE", "LONGITUDE", "AC_MASS", "SPEED"]
categorical_features = ["AC_CLASS", "TYPE_ENG", "SPECIES_ID", "SIZE", "NUM_STRUCK"]
feature_columns = [col for col in selected_columns if col != "INDICATED_DAMAGE"]


def clean_feature_frame(df, columns):
    # Copy only the columns we need so later cleaning does not affect the original dataframe.
    cleaned = df[columns].copy()

    # Convert numeric columns to numbers. Any invalid value becomes NaN.
    for col in numeric_features:
        if col in cleaned.columns:
            cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

    # Treat blank categorical values as missing so the imputer can handle them consistently.
    for col in categorical_features:
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].replace(r"^\s*$", np.nan, regex=True)

    return cleaned


def prepare_training_data(df):
    # Clean the training columns first so missing and malformed values are standardized.
    cleaned = clean_feature_frame(df, selected_columns)

    # Drop rows only when the target is missing. Feature NaNs will be handled in the model pipeline.
    cleaned = cleaned.dropna(subset=["INDICATED_DAMAGE"]).copy()
    return cleaned


def prepare_prediction_data(df):
    # Clean the test features but keep rows with missing feature values for the imputer to fill.
    return clean_feature_frame(df, feature_columns)


# Build the cleaned training dataset that will be used for splitting and fitting.
train_clean = prepare_training_data(train)

# Take a quick look at the cleaned training data.
print(train_clean.shape)
print(train_clean.head())
print(train_clean.dtypes)
        


(307178, 10)
    LATITUDE   LONGITUDE AC_CLASS  AC_MASS TYPE_ENG  SPEED SPECIES_ID    SIZE  \
0  18.439420  -66.001830      A        4.0        D  145.0      UNKBS   Small   
1   2.745578  101.709917      A        4.0        D    NaN      UNKBM  Medium   
2  38.174390  -85.736000      A        4.0        D  240.0      UNKBL   Large   
3  33.942540 -118.408070      NaN      NaN      NaN    NaN      NE120  Medium   
4  21.975980 -159.338960      A        4.0        D  135.0      R1101  Medium   

  NUM_STRUCK  INDICATED_DAMAGE  
0     10-Feb                 0  
1          1                 0  
2          1                 1  
3     10-Feb                 0  
4          1                 0  
LATITUDE            float64
LONGITUDE           float64
AC_CLASS             object
AC_MASS             float64
TYPE_ENG             object
SPEED               float64
SPECIES_ID           object
SIZE                 object
NUM_STRUCK           object
INDICATED_DAMAGE      int64
dtype: object


## Decision Tree Baseline

We first train a `Decision Tree` as the baseline classifier.

Because the competition is scored with balanced accuracy, we report that metric alongside accuracy, precision, recall, F1, and ROC AUC.
        


In [11]:
# Split the cleaned data into features and target
X = train_clean.drop(columns="INDICATED_DAMAGE")
y = train_clean["INDICATED_DAMAGE"]

# Hold out part of the training data for evaluation
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Fill missing numeric values with the median from the training data.
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

# Fill missing categorical values with the most common category, then one-hot encode them.
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# Apply the numeric and categorical preprocessing so train, validation, and test are handled the same way.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Build the decision tree pipeline so preprocessing and prediction stay connected.
decision_tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, class_weight="balanced", random_state=42)),
    ]
)

# Fit the decision tree on the training split.
decision_tree_model.fit(X_train, y_train)

# Predict on the validation split to measure model quality.
dt_val_predictions = decision_tree_model.predict(X_val)
dt_val_probabilities = decision_tree_model.predict_proba(X_val)[:, 1]

# Print the main evaluation metrics for the decision tree, including balanced accuracy for the competition.
print("Decision Tree Evaluation Results")
print(f"Accuracy:           {accuracy_score(y_val, dt_val_predictions):.4f}")
print(f"Balanced Accuracy:  {balanced_accuracy_score(y_val, dt_val_predictions):.4f}")
print(f"Precision:          {precision_score(y_val, dt_val_predictions):.4f}")
print(f"Recall:             {recall_score(y_val, dt_val_predictions):.4f}")
print(f"F1 Score:           {f1_score(y_val, dt_val_predictions):.4f}")
print(f"ROC AUC:            {roc_auc_score(y_val, dt_val_probabilities):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, dt_val_predictions))
print("\nClassification Report:")
print(classification_report(y_val, dt_val_predictions))

# Retrain the tree on all cleaned training rows before making final test predictions.
decision_tree_model.fit(X, y)

# Load the test data and keep every row so the model predicts on the full submission set.
test_data = pd.read_csv("test.csv", low_memory=False)
X_test = prepare_prediction_data(test_data)

# Generate predictions for every test row after the pipeline fills missing values.
dt_test_predictions = decision_tree_model.predict(X_test)

# Build and save the decision tree submission file.
decision_tree_submission = test_data[["INDEX_NR"]].copy()
decision_tree_submission["INDICATED_DAMAGE"] = dt_test_predictions
decision_tree_submission.to_csv("decision_tree_submission.csv", index=False)
print("Saved submission file to decision_tree_submission.csv")
        


Decision Tree Evaluation Results
Accuracy:           0.7601
Balanced Accuracy:  0.7708
Precision:          0.1804
Recall:             0.7831
F1 Score:           0.2932
ROC AUC:            0.8514

Confusion Matrix:
[[43637 13894]
 [  847  3058]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.76      0.86     57531
           1       0.18      0.78      0.29      3905

    accuracy                           0.76     61436
   macro avg       0.58      0.77      0.57     61436
weighted avg       0.93      0.76      0.82     61436

Saved submission file to decision_tree_submission.csv


## Random Forest Results

We then chose to ensemble the decision tree to create a `Random Forest`, hoping to reduce variance and get a better result. However, it turned out that the Balanced Accuracy seemed to come out worse. This likely means that we overtrained the model to the training data and it is not making generalizations well (Too complex of a decision boundary)

This result is still useful because it shows that a more complex methods are not automatically better for the competition.

In [12]:
# Build the random forest pipeline using the same preprocessing steps as the decision tree.
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=16,
                min_samples_leaf=10,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

# Fit the random forest on the training split.
random_forest_model.fit(X_train, y_train)

# Predict on the validation split to compare performance.
rf_val_predictions = random_forest_model.predict(X_val)
rf_val_probabilities = random_forest_model.predict_proba(X_val)[:, 1]

# Print the main evaluation metrics for the random forest, including balanced accuracy for the competition.
print("Random Forest Evaluation Results")
print(f"Accuracy:           {accuracy_score(y_val, rf_val_predictions):.4f}")
print(f"Balanced Accuracy:  {balanced_accuracy_score(y_val, rf_val_predictions):.4f}")
print(f"Precision:          {precision_score(y_val, rf_val_predictions):.4f}")
print(f"Recall:             {recall_score(y_val, rf_val_predictions):.4f}")
print(f"F1 Score:           {f1_score(y_val, rf_val_predictions):.4f}")
print(f"ROC AUC:            {roc_auc_score(y_val, rf_val_probabilities):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, rf_val_predictions))
print("\nClassification Report:")
print(classification_report(y_val, rf_val_predictions))

# Retrain the random forest on all cleaned training rows.
random_forest_model.fit(X, y)

# Generate predictions for every test row using the same imputation pipeline.
rf_test_predictions = random_forest_model.predict(X_test)

# Build and save the random forest submission file.
random_forest_submission = test_data[["INDEX_NR"]].copy()
random_forest_submission["INDICATED_DAMAGE"] = rf_test_predictions
random_forest_submission.to_csv("random_forest_submission.csv", index=False)
print("Saved submission file to random_forest_submission.csv")
        


Random Forest Evaluation Results
Accuracy:           0.7242
Balanced Accuracy:  0.7620
Precision:          0.1627
Recall:             0.8054
F1 Score:           0.2707
ROC AUC:            0.8516

Confusion Matrix:
[[41344 16187]
 [  760  3145]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.72      0.83     57531
           1       0.16      0.81      0.27      3905

    accuracy                           0.72     61436
   macro avg       0.57      0.76      0.55     61436
weighted avg       0.93      0.72      0.79     61436

Saved submission file to random_forest_submission.csv


## Export Submission Files

This final code cell copies the newly created submission CSV files so they are available outside the notebook workflow and ready to upload for the competition submission.


In [16]:
# Get files from docker container onto local machine for submission
!cp decision_tree_submission.csv .
!cp random_forest_submission.csv .

cp: 'decision_tree_submission.csv' and './decision_tree_submission.csv' are the same file
cp: 'random_forest_submission.csv' and './random_forest_submission.csv' are the same file
